# 03 — Small DL Model + ONNX Export (T132)

1D-CNN intent classifier with learned word embeddings, exported to ONNX.
Reads `data/clinc150_mapped.csv` produced by T130; trains on `split == "train"`,
early-stops on `split == "val"`, evaluates on `split == "test"`. No leakage.

**Constitution note** (Principle IV — Lean Containers): torch is used here in
the *training* notebook only. The modelserver serving container ships the
ONNX artifact and runs it via `onnxruntime` — no torch in the image.

**Outputs**:
* `services/modelserver/artifacts/cnn_intent.onnx` — exported model
* `services/modelserver/artifacts/cnn_vocab.json` — token→id map + max_len
* `notebooks/results/cnn_onnx_results.json` — metrics + artifact SHA-256

In [1]:
from __future__ import annotations

import hashlib
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset

LABELS: tuple[str, ...] = ("spam", "faq", "lead_intent", "escalate", "ambiguous")
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

DATA_CSV = Path("data/clinc150_mapped.csv")
ARTIFACT_PATH = Path("../services/modelserver/artifacts/cnn_intent.onnx")
VOCAB_PATH = Path("../services/modelserver/artifacts/cnn_vocab.json")
RESULTS_PATH = Path("results/cnn_onnx_results.json")

MAX_VOCAB = 20000
MIN_FREQ = 2
MAX_LEN = 32
EMBED_DIM = 64
NUM_FILTERS = 128
KERNEL_SIZE = 3
DROPOUT = 0.3
LR = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 10
PATIENCE = 3

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")  # constitution: no GPU runtime
print(f"torch={torch.__version__}  onnxruntime={ort.__version__}  device={DEVICE}")

torch=2.12.0+cpu  onnxruntime=1.26.0  device=cpu


In [2]:
df = pd.read_csv(DATA_CSV)
train = df[df["split"] == "train"].reset_index(drop=True)
val = df[df["split"] == "val"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
print(f"train: {len(train):5d}   val: {len(val):5d}   test: {len(test):5d}")

train:  4550   val:   960   test:  2290


In [3]:
# Simple whitespace + punctuation tokenizer. Each word-char run is one token,
# each punctuation mark is its own token. Lowercased.
_TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())

# Sanity check on a few rows.
for sample in train["text"].head(3):
    print(f"{sample!r}\n  -> {tokenize(sample)}")

'what is the meaning of realism'
  -> ['what', 'is', 'the', 'meaning', 'of', 'realism']
'what is regard mean'
  -> ['what', 'is', 'regard', 'mean']
'what is the meaning of interorganizational'
  -> ['what', 'is', 'the', 'meaning', 'of', 'interorganizational']


In [4]:
# (1) Build vocab from train split only (no leakage from val/test).
counter: Counter[str] = Counter()
for text in train["text"]:
    counter.update(tokenize(text))

filtered = [(tok, freq) for tok, freq in counter.most_common() if freq >= MIN_FREQ]
kept = filtered[: MAX_VOCAB - 2]  # leave room for <pad> and <unk>

vocab: dict[str, int] = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for tok, _ in kept:
    vocab[tok] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"Vocab size: {VOCAB_SIZE} (cap {MAX_VOCAB}, min_freq {MIN_FREQ})")
print(f"Most common 10: {counter.most_common(10)}")
print(f"Tokens with freq < {MIN_FREQ}: {sum(1 for _, f in counter.items() if f < MIN_FREQ)} (mapped to <unk>)")

Vocab size: 1423 (cap 20000, min_freq 2)
Most common 10: [('i', 1387), ('my', 1235), ('to', 1179), ('the', 1164), ('what', 1033), ('you', 984), ('a', 940), ('me', 777), ('for', 746), ('is', 698)]
Tokens with freq < 2: 1471 (mapped to <unk>)


In [5]:
def encode(text: str) -> list[int]:
    """Tokenize → ids, pad/truncate to MAX_LEN."""
    ids = [vocab.get(tok, UNK_ID) for tok in tokenize(text)][:MAX_LEN]
    if len(ids) < MAX_LEN:
        ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids

def make_tensors(frame: pd.DataFrame) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.tensor([encode(t) for t in frame["text"]], dtype=torch.long)
    y = torch.tensor([LABEL_TO_ID[lbl] for lbl in frame["label"]], dtype=torch.long)
    return X, y

X_train, y_train = make_tensors(train)
X_val, y_val = make_tensors(val)
X_test, y_test = make_tensors(test)
print(f"X_train: {tuple(X_train.shape)}  X_val: {tuple(X_val.shape)}  X_test: {tuple(X_test.shape)}")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)

X_train: (4550, 32)  X_val: (960, 32)  X_test: (2290, 32)

In [6]:
# (2) Architecture per spec:
# Embedding(V, 64) -> Conv1d(64, 128, k=3) -> GlobalMaxPool -> Dropout(0.3) -> Linear(128, 5)
class IntentCNN(nn.Module):
    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_ID)
        # padding=1 so kernel=3 keeps temporal length stable.
        self.conv = nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=KERNEL_SIZE, padding=1)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(NUM_FILTERS, len(LABELS))

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # input_ids: (B, T)
        x = self.embedding(input_ids)         # (B, T, E)
        x = x.transpose(1, 2)                  # (B, E, T) for Conv1d
        x = F.relu(self.conv(x))               # (B, F, T)
        x, _ = x.max(dim=2)                    # GlobalMaxPool: (B, F)
        x = self.dropout(x)
        return self.fc(x)                      # (B, C)

model = IntentCNN(VOCAB_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters: {n_params:,}")

IntentCNN(
  (embedding): Embedding(1423, 64, padding_idx=0)
  (conv): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

Parameters: 116,421


In [7]:
# Training loop with early stopping on val macro-F1.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

def eval_macro_f1(loader: DataLoader) -> tuple[float, list[int]]:
    model.eval()
    preds: list[int] = []
    truths: list[int] = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            preds.extend(logits.argmax(dim=1).cpu().tolist())
            truths.extend(yb.tolist())
    return float(f1_score(truths, preds, labels=list(range(len(LABELS))), average="macro")), preds

best_val_f1 = -1.0
best_state: dict[str, torch.Tensor] | None = None
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_loader.dataset)

    val_f1, _ = eval_macro_f1(val_loader)
    improved = val_f1 > best_val_f1
    print(f"epoch {epoch:2d}  train_loss={epoch_loss:.4f}  val_macro_f1={val_f1:.4f}" + ("  *" if improved else ""))
    if improved:
        best_val_f1 = val_f1
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"early-stop after {epoch} epochs (no val improvement for {PATIENCE})")
            break

assert best_state is not None
model.load_state_dict(best_state)
print(f"\nLoaded best checkpoint (val macro-F1 = {best_val_f1:.4f}).")

epoch  1  train_loss=1.1089  val_macro_f1=0.6373  *


epoch  2  train_loss=0.6341  val_macro_f1=0.7649  *


epoch  3  train_loss=0.4377  val_macro_f1=0.8260  *


epoch  4  train_loss=0.3293  val_macro_f1=0.8471  *


epoch  5  train_loss=0.2582  val_macro_f1=0.8687  *


epoch  6  train_loss=0.1987  val_macro_f1=0.8776  *


epoch  7  train_loss=0.1664  val_macro_f1=0.8866  *


epoch  8  train_loss=0.1397  val_macro_f1=0.8909  *


epoch  9  train_loss=0.1218  val_macro_f1=0.8930  *


epoch 10  train_loss=0.0958  val_macro_f1=0.8954  *

Loaded best checkpoint (val macro-F1 = 0.8954).


In [8]:
# (3) Evaluate best checkpoint on held-out test split.
test_macro_f1, test_preds = eval_macro_f1(test_loader)
test_truths = y_test.tolist()

per_class_f1_arr = f1_score(test_truths, test_preds, labels=list(range(len(LABELS))), average=None)
per_class_f1 = {ID_TO_LABEL[i]: float(score) for i, score in enumerate(per_class_f1_arr)}

print(f"Test macro-F1: {test_macro_f1:.4f}\n")
print("Per-class F1:")
for label, score in per_class_f1.items():
    print(f"  {label:12s} {score:.4f}")
print()
print(classification_report(
    test_truths, test_preds,
    labels=list(range(len(LABELS))), target_names=list(LABELS), digits=4,
))

Test macro-F1: 0.8267

Per-class F1:
  spam         0.7523
  faq          0.7788
  lead_intent  0.8042
  escalate     0.9052
  ambiguous    0.8931

              precision    recall  f1-score   support

        spam     0.9685    0.6150    0.7523      1000
         faq     0.6541    0.9622    0.7788       450
 lead_intent     0.6957    0.9528    0.8042       360
    escalate     0.8754    0.9370    0.9052       270
   ambiguous     0.8910    0.8952    0.8931       210

    accuracy                         0.8000      2290
   macro avg     0.8170    0.8725    0.8267      2290
weighted avg     0.8458    0.8000    0.7966      2290



In [9]:
cm = confusion_matrix(test_truths, test_preds, labels=list(range(len(LABELS))))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{lbl}" for lbl in LABELS],
    columns=[f"pred_{lbl}" for lbl in LABELS],
)
print(cm_df)

                  pred_spam  pred_faq  pred_lead_intent  pred_escalate  \
true_spam               615       202               134             31   
true_faq                 10       433                 7              0   
true_lead_intent          4        10               343              1   
true_escalate             6         3                 5            253   
true_ambiguous            0        14                 4              4   

                  pred_ambiguous  
true_spam                     18  
true_faq                       0  
true_lead_intent               2  
true_escalate                  3  
true_ambiguous               188  


In [10]:
# (4) Prepare the 1000 single-prediction latency samples.
# Latency is measured after ONNX export below, because production serves this
# model with onnxruntime rather than PyTorch.
model.eval()
sample_texts = test["text"].sample(1000, replace=True, random_state=SEED).tolist()
sample_ids = torch.tensor([encode(t) for t in sample_texts], dtype=torch.long)
print(f"Prepared {len(sample_texts)} latency samples for ONNX runtime timing.")

Prepared 1000 latency samples for ONNX runtime timing.


In [11]:
# (5) Export to ONNX with dynamic batch axis.
# torch 2.12's default exporter (dynamo=True) requires onnxscript; the legacy
# tracer-based path is sufficient for a small CNN with no control flow, so
# we pin dynamo=False to keep the notebook dependency surface small.
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.eval()
dummy = torch.zeros((1, MAX_LEN), dtype=torch.long)

torch.onnx.export(
    model,
    (dummy,),
    str(ARTIFACT_PATH),
    input_names=["input_ids"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
size_kb = ARTIFACT_PATH.stat().st_size / 1024
print(f"Exported ONNX: {ARTIFACT_PATH.resolve()}")
print(f"Size: {size_kb:,.1f} KB")

C:\Users\ahmad\AppData\Local\Temp\ipykernel_22440\1510884505.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported ONNX: C:\Users\ahmad\OneDrive\Desktop\AIE bootcamp\week8_project\CONCIERGE\services\modelserver\artifacts\cnn_intent.onnx
Size: 455.6 KB


In [12]:
# (6) Parity check + production-style latency with onnxruntime.
# The same 1000 samples are timed one-by-one to match the modelserver's
# visitor-message path.
session = ort.InferenceSession(str(ARTIFACT_PATH), providers=["CPUExecutionProvider"])
sample_np = sample_ids.numpy().astype(np.int64)

with torch.no_grad():
    pt_logits = model(sample_ids).numpy()
pt_preds = pt_logits.argmax(axis=1)

start = time.perf_counter()
ort_single_logits: list[np.ndarray] = []
for i in range(len(sample_np)):
    ort_single_logits.append(session.run(["logits"], {"input_ids": sample_np[i : i + 1]})[0])
elapsed_s = time.perf_counter() - start
latency_ms = (elapsed_s / len(sample_np)) * 1000

ort_logits = np.concatenate(ort_single_logits, axis=0)
ort_preds = ort_logits.argmax(axis=1)

match_count = int((pt_preds == ort_preds).sum())
match_rate = match_count / len(sample_texts)
print(f"PyTorch / ONNX prediction match: {match_count}/{len(sample_texts)} = {match_rate:.4f}")
print(f"ONNX total time for {len(sample_texts)} predictions: {elapsed_s:.3f}s")
print(f"ONNX mean per prediction: {latency_ms:.3f} ms")
assert match_rate >= 0.99, f"ONNX parity below threshold: {match_rate:.4f}"
print("OK: parity >= 99%")

PyTorch / ONNX prediction match: 1000/1000 = 1.0000
ONNX total time for 1000 predictions: 0.083s
ONNX mean per prediction: 0.083 ms
OK: parity >= 99%


In [13]:
# Save vocab + max_len for the modelserver to mirror the same encoding at serve time.
VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
VOCAB_PATH.write_text(
    json.dumps({"vocab": vocab, "max_len": MAX_LEN, "pad_id": PAD_ID, "unk_id": UNK_ID}, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Saved vocab to {VOCAB_PATH.resolve()} ({VOCAB_PATH.stat().st_size:,} bytes)")

# (7) SHA-256 of the ONNX artifact (modelserver verifies at boot — T148).
artifact_sha256 = hashlib.sha256(ARTIFACT_PATH.read_bytes()).hexdigest()
vocab_sha256 = hashlib.sha256(VOCAB_PATH.read_bytes()).hexdigest()
print(f"Artifact SHA-256: {artifact_sha256}")
print(f"Vocab    SHA-256: {vocab_sha256}")

Saved vocab to C:\Users\ahmad\OneDrive\Desktop\AIE bootcamp\week8_project\CONCIERGE\services\modelserver\artifacts\cnn_vocab.json (21,120 bytes)
Artifact SHA-256: 6cfbc65825235efc576a35dec062a116078cd229dad82bddf7c402db6fabe437
Vocab    SHA-256: ac00cac61e2f8fce37607cde73bb3ba65a643015e4b99dc5ce8ecaf049bc0996


In [14]:
# (8) Results dict consumed by T134 (compare/export).
results = {
    "model": "cnn_onnx",
    "macro_f1": float(test_macro_f1),
    "per_class_f1": per_class_f1,
    "latency_ms_per_prediction": float(latency_ms),
    "latency_runtime": "onnxruntime_cpu",
    "cost_per_1k_predictions": 0.00,  # local inference, no API calls
    "artifact_path": str(ARTIFACT_PATH).replace("\\", "/"),
    "artifact_sha256": artifact_sha256,
    "vocab_path": str(VOCAB_PATH).replace("\\", "/"),
    "vocab_sha256": vocab_sha256,
    "onnx_match_rate": float(match_rate),
}
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))

{
  "model": "cnn_onnx",
  "macro_f1": 0.8267180858734763,
  "per_class_f1": {
    "spam": 0.7522935779816514,
    "faq": 0.7787769784172662,
    "lead_intent": 0.8042203985932005,
    "escalate": 0.9051878354203936,
    "ambiguous": 0.8931116389548693
  },
  "latency_ms_per_prediction": 0.08294809999642894,
  "latency_runtime": "onnxruntime_cpu",
  "cost_per_1k_predictions": 0.0,
  "artifact_path": "../services/modelserver/artifacts/cnn_intent.onnx",
  "artifact_sha256": "6cfbc65825235efc576a35dec062a116078cd229dad82bddf7c402db6fabe437",
  "vocab_path": "../services/modelserver/artifacts/cnn_vocab.json",
  "vocab_sha256": "ac00cac61e2f8fce37607cde73bb3ba65a643015e4b99dc5ce8ecaf049bc0996",
  "onnx_match_rate": 1.0
}


## Next

`04_llm_zero_shot.ipynb` (T133) runs Claude as a zero-shot classifier on the
same test split. `05_compare_and_export.ipynb` (T134) reads all three results
JSONs (tfidf, cnn, llm), compares by macro-F1 with latency / size / cost
tiebreakers, and copies the winning artifact into
`services/modelserver/artifacts/model.{onnx,joblib}` plus an updated
`model_card.yaml`.